# ETL vs ELT — Batch vs Streaming
### A hands-on class notebook using Python, pandas, and SQL (SQLite)

**Agenda**
1. Setup — create a messy sample dataset
2. ETL (batch) — transform in pandas, then load
3. ELT (batch) — load raw, transform with SQL
4. Streaming — simulate live events, ETL-style
5. Compare results side by side
6. Class exercise


## 1. Setup

We'll generate a small, deliberately messy `orders_raw.csv` — duplicate IDs, missing amounts,
mixed-up rows — so the cleaning step actually does something visible.


In [1]:
import pandas as pd
import numpy as np
import sqlite3
import time
import random
from datetime import datetime

# Make results repeatable
random.seed(42)
np.random.seed(42)


In [2]:
# Create a messy raw dataset and save it as CSV, just like data you'd receive from
# an upstream system (an app, an API export, another team, etc.)

raw_data = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 104, 105, 106, 107, 108, 109],   # 104 is duplicated
    "order_date": ["2024-01-01", "2024-01-02", "2024-01-02", "2024-01-03",
                   "2024-01-03", "2024-01-04", "2024-01-04", "2024-01-05",
                   "2024-01-05", "2024-01-06"],
    "amount": [25.50, np.nan, 40.00, 15.75, 15.75, -5.00, 60.00, np.nan, 33.20, 12.00]
    # np.nan = missing amount, -5.00 = bad/invalid amount
})

raw_data.to_csv("orders_raw.csv", index=False)
raw_data


,order_id,order_date,amount
0,101,2024-01-01,25.50
1,102,2024-01-02,NaN
2,103,2024-01-02,40.00
3,104,2024-01-03,15.75
4,104,2024-01-03,15.75
5,105,2024-01-04,-5.00
6,106,2024-01-04,60.00
7,107,2024-01-05,NaN
8,108,2024-01-05,33.20
9,109,2024-01-06,12.00


## 2. ETL — Extract, Transform, Load (batch)

**Idea:** clean the data in pandas *before* it goes into the database.
By the time it lands in the destination table, it's already trustworthy.


In [3]:
# EXTRACT — read the raw file
raw = pd.read_csv("orders_raw.csv")
print("Extracted rows:", len(raw))
raw


Extracted rows: 10


,order_id,order_date,amount
0,101,2024-01-01,25.50
1,102,2024-01-02,NaN
2,103,2024-01-02,40.00
3,104,2024-01-03,15.75
4,104,2024-01-03,15.75
5,105,2024-01-04,-5.00
6,106,2024-01-04,60.00
7,107,2024-01-05,NaN
8,108,2024-01-05,33.20
9,109,2024-01-06,12.00


In [4]:
# TRANSFORM — clean and reshape in pandas, BEFORE loading
df = raw.copy()

df["order_date"] = pd.to_datetime(df["order_date"])   # fix types
df["amount"] = df["amount"].fillna(0)                 # fill missing amounts
df = df[df["amount"] > 0]                              # drop invalid/negative amounts
df = df.drop_duplicates(subset="order_id")              # remove duplicate orders
df["amount_usd"] = (df["amount"] * 0.012).round(2)      # example: INR -> USD conversion

print("Rows after cleaning:", len(df))
df


Rows after cleaning: 6


,order_id,order_date,amount,amount_usd
0,101,2024-01-01,25.50,0.31
2,103,2024-01-02,40.00,0.48
3,104,2024-01-03,15.75,0.19
6,106,2024-01-04,60.00,0.72
8,108,2024-01-05,33.20,0.40
9,109,2024-01-06,12.00,0.14


In [5]:
# LOAD — write the ALREADY-CLEAN data into the destination (SQLite here,
# but could just as easily be Postgres, Snowflake, BigQuery, etc.)
conn = sqlite3.connect("warehouse.db")
df.to_sql("orders_clean_etl", conn, if_exists="replace", index=False)

pd.read_sql("SELECT * FROM orders_clean_etl", conn)


,order_id,order_date,amount,amount_usd
0,101,2024-01-01 00:00:00,25.50,0.31
1,103,2024-01-02 00:00:00,40.00,0.48
2,104,2024-01-03 00:00:00,15.75,0.19
3,106,2024-01-04 00:00:00,60.00,0.72
4,108,2024-01-05 00:00:00,33.20,0.40
5,109,2024-01-06 00:00:00,12.00,0.14


**Key point:** in ETL, the *only* table that exists downstream is the clean one.
The messy raw data was fixed in pandas and never touched the database.


## 3. ELT — Extract, Load, Transform (batch)

**Idea:** load the raw data as-is first, then clean it up *inside* the database using SQL.
This keeps a permanent copy of the raw data for auditing, and pushes the transform work
to the database engine (which is often faster at big joins/aggregations than pandas).


In [6]:
# EXTRACT
raw = pd.read_csv("orders_raw.csv")

conn = sqlite3.connect("warehouse.db")

# LOAD — dump the RAW data as-is, no cleaning at all
raw.to_sql("orders_raw_elt", conn, if_exists="replace", index=False)

pd.read_sql("SELECT * FROM orders_raw_elt", conn)


,order_id,order_date,amount
0,101,2024-01-01,25.50
1,102,2024-01-02,NaN
2,103,2024-01-02,40.00
3,104,2024-01-03,15.75
4,104,2024-01-03,15.75
5,105,2024-01-04,-5.00
6,106,2024-01-04,60.00
7,107,2024-01-05,NaN
8,108,2024-01-05,33.20
9,109,2024-01-06,12.00


In [7]:
# TRANSFORM — cleaning now happens AFTER loading, using SQL inside the database
cur = conn.cursor()
cur.execute("DROP TABLE IF EXISTS orders_clean_elt")
cur.execute('''
CREATE TABLE orders_clean_elt AS
SELECT
    order_id,
    DATE(order_date) AS order_date,
    COALESCE(amount, 0) AS amount,
    ROUND(COALESCE(amount, 0) * 0.012, 2) AS amount_usd
FROM orders_raw_elt
WHERE COALESCE(amount, 0) > 0
GROUP BY order_id
''')
conn.commit()

pd.read_sql("SELECT * FROM orders_clean_elt", conn)


,order_id,order_date,amount,amount_usd
0,101,2024-01-01,25.50,0.31
1,103,2024-01-02,40.00,0.48
2,104,2024-01-03,15.75,0.19
3,106,2024-01-04,60.00,0.72
4,108,2024-01-05,33.20,0.40
5,109,2024-01-06,12.00,0.14


**Key point:** notice `orders_raw_elt` still exists in the database, untouched.
That's the defining trait of ELT — raw data is preserved, and transformation is
just another query you can re-run, tweak, or audit later.


## 4. Batch vs Streaming

So far both examples were **batch**: we processed the whole file at once, then stopped.

**Streaming** processes each record (or small micro-batch) continuously, as it arrives.
Below we simulate a live stream of events with a Python generator — same transform logic
as before, just applied one record at a time instead of to a full DataFrame.

In real systems, this loop would be replaced by something like a Kafka consumer or
Spark Structured Streaming — but the underlying logic is identical, which is the point.


In [8]:
def event_stream(n=8):
    """Simulates a live stream of incoming order events."""
    for _ in range(n):
        yield {
            "order_id": random.randint(1000, 9999),
            "amount": round(random.uniform(-10, 200), 2),  # sometimes invalid (negative)
            "timestamp": datetime.now()
        }
        time.sleep(0.3)  # pretend a new event arrives every 0.3s


In [9]:
# STREAMING ETL: transform + load each event AS it arrives
conn = sqlite3.connect("warehouse.db")
cur = conn.cursor()
cur.execute('''
CREATE TABLE IF NOT EXISTS orders_streaming (
    order_id INTEGER,
    amount REAL,
    amount_usd REAL,
    event_time TEXT
)
''')
conn.commit()

for event in event_stream(n=8):
    # transform (same rule as batch: drop invalid amounts, convert currency)
    if event["amount"] <= 0:
        print(f"Skipped invalid event: {event}")
        continue

    event["amount_usd"] = round(event["amount"] * 0.012, 2)

    # load — insert this single event immediately
    cur.execute(
        "INSERT INTO orders_streaming VALUES (?, ?, ?, ?)",
        (event["order_id"], event["amount"], event["amount_usd"], str(event["timestamp"]))
    )
    conn.commit()
    print(f"Loaded event: {event}")

pd.read_sql("SELECT * FROM orders_streaming", conn)


Skipped invalid event: {'order_id': 2824, 'amount': -4.75, 'timestamp': datetime.datetime(2026, 7, 20, 9, 25, 40, 420232)}
Loaded event: {'order_id': 5506, 'amount': 41.43, 'timestamp': datetime.datetime(2026, 7, 20, 9, 25, 40, 720561), 'amount_usd': 0.5}
Loaded event: {'order_id': 3286, 'amount': 144.66, 'timestamp': datetime.datetime(2026, 7, 20, 9, 25, 41, 28981), 'amount_usd': 1.74}
Loaded event: {'order_id': 9935, 'amount': 8.26, 'timestamp': datetime.datetime(2026, 7, 20, 9, 25, 41, 338049), 'amount_usd': 0.1}
Skipped invalid event: {'order_id': 7912, 'amount': -3.33, 'timestamp': datetime.datetime(2026, 7, 20, 9, 25, 41, 646688)}
Loaded event: {'order_id': 2535, 'amount': 35.91, 'timestamp': datetime.datetime(2026, 7, 20, 9, 25, 41, 946926), 'amount_usd': 0.43}
Loaded event: {'order_id': 9279, 'amount': 116.42, 'timestamp': datetime.datetime(2026, 7, 20, 9, 25, 42, 255023), 'amount_usd': 1.4}
Loaded event: {'order_id': 4257, 'amount': 140.36, 'timestamp': datetime.datetime(2026,

,order_id,amount,amount_usd,event_time
0,5506,41.43,0.50,2026-07-20 09:25:40.720561
1,3286,144.66,1.74,2026-07-20 09:25:41.028981
2,9935,8.26,0.10,2026-07-20 09:25:41.338049
3,2535,35.91,0.43,2026-07-20 09:25:41.946926
4,9279,116.42,1.40,2026-07-20 09:25:42.255023
5,4257,140.36,1.68,2026-07-20 09:25:42.563010


**Discussion for class:** the transform rule (`amount > 0`) is identical to the batch
ETL example. The only thing that changed is *when* it runs — per record, instead of on
a full DataFrame at the end of the day.


## 5. Compare all outputs side by side


In [10]:
conn = sqlite3.connect("warehouse.db")

print("ETL result (clean-before-load):")
display(pd.read_sql("SELECT * FROM orders_clean_etl", conn))

print("\nELT result (clean-after-load):")
display(pd.read_sql("SELECT * FROM orders_clean_elt", conn))

print("\nStreaming result (clean-as-it-arrives):")
display(pd.read_sql("SELECT * FROM orders_streaming", conn))


ETL result (clean-before-load):


,order_id,order_date,amount,amount_usd
0,101,2024-01-01 00:00:00,25.50,0.31
1,103,2024-01-02 00:00:00,40.00,0.48
2,104,2024-01-03 00:00:00,15.75,0.19
3,106,2024-01-04 00:00:00,60.00,0.72
4,108,2024-01-05 00:00:00,33.20,0.40
5,109,2024-01-06 00:00:00,12.00,0.14



ELT result (clean-after-load):


,order_id,order_date,amount,amount_usd
0,101,2024-01-01,25.50,0.31
1,103,2024-01-02,40.00,0.48
2,104,2024-01-03,15.75,0.19
3,106,2024-01-04,60.00,0.72
4,108,2024-01-05,33.20,0.40
5,109,2024-01-06,12.00,0.14



Streaming result (clean-as-it-arrives):


,order_id,amount,amount_usd,event_time
0,5506,41.43,0.50,2026-07-20 09:25:40.720561
1,3286,144.66,1.74,2026-07-20 09:25:41.028981
2,9935,8.26,0.10,2026-07-20 09:25:41.338049
3,2535,35.91,0.43,2026-07-20 09:25:41.946926
4,9279,116.42,1.40,2026-07-20 09:25:42.255023
5,4257,140.36,1.68,2026-07-20 09:25:42.563010


| | Batch | Streaming |
|---|---|---|
| **ETL** | pandas cleans data, then loads into DB | clean each event on arrival, then insert |
| **ELT** | dump raw CSV into DB, run SQL transform once | dump raw events continuously, run SQL transform periodically |


## 6. Class exercise

Using `orders_raw.csv` above (or a messier version you create yourselves):

1. **ETL team:** clean the data fully in pandas, then load only the clean table.
2. **ELT team:** load the raw data untouched, then write a `CREATE TABLE AS SELECT`
   query to clean it inside SQLite.

**Discuss:**
- Which approach was easier to debug when something looked wrong?
- Which approach keeps more of the original data around for auditing?
- If this dataset were 500 GB instead of 10 rows, which approach would you prefer, and why?
- Which approach maps more naturally onto the streaming example — why?
